# Exploring byte-level BPE on FineWeb2

Run these cells yourself to train a small tokenizer and inspect its pipeline. No results have been generated.

References: [Tokenizers v0.22.2](https://github.com/huggingface/tokenizers/tree/v0.22.2), [byte-level BPE implementation](https://github.com/huggingface/tokenizers/blob/v0.22.2/bindings/python/py_src/tokenizers/implementations/byte_level_bpe.py), [FineWeb2](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2), [streaming guide](https://huggingface.co/docs/datasets/en/stream).

FineWeb2 provides non-English language-script subsets. This example uses Russian (`rus_Cyrl`); French (`fra_Latn`) and German (`deu_Latn`) are alternatives. English is in the original FineWeb.


## Environment

Uncomment the install line to install into this notebook's kernel. Restart the kernel afterwards if needed. The pinned stable version provides the training API used here.


In [5]:
# %pip install "tokenizers==0.22.2" datasets

In [10]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

In [2]:
import json
from importlib.metadata import version

from datasets import load_dataset
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

print({package: version(package) for package in ("tokenizers", "datasets")})


{'tokenizers': '0.22.2', 'datasets': '5.1.0'}


## Stream a small sample

Start with 100 documents, then increase the sample once the pipeline works. Streaming avoids downloading the entire corpus. The finite shuffle buffer produces an approximate shuffle. We keep the small sample in memory so retraining uses the same texts.


In [9]:
dataset_id = "HuggingFaceFW/fineweb-2"
language = "rus_Cyrl"
num_documents = 100
vocab_size = 8_000
seed = 42

corpus = load_dataset(dataset_id, name=language, split="train", streaming=True)
sample = corpus.shuffle(seed=seed, buffer_size=100).take(num_documents)
texts = [document["text"] for document in sample]
assert texts, "No documents returned."

print("Documents:", len(texts))
print("UTF-8 bytes:", sum(len(text.encode("utf-8")) for text in texts))
print(texts[0][:500])


Resolving data files:   0%|          | 0/440 [00:00<?, ?it/s]

Documents: 100
UTF-8 bytes: 784056
Относительно вероятности возникновения аварийной ситуации подсчитано, что на скорости 80 км/ч она почти вдвое выше, чем при предельно допустимой в городских условиях. Примечательно, что превышение скорости не дает практически никакого выигрыша во времени: уйдя рывком с перекрестка, водитель, как правило, останавливается у следующего пересечения улиц перед красным сигналом светофора. В то же время те, кто выбрал скорость с расчетом, чтобы приблизиться к перекрестку в момент включения зеленого све


## Inspect pre-tokenization

This step splits the input before BPE applies merges and represents UTF-8 bytes with a reversible alphabet. Displayed symbols can look different from the original characters. No normalization is applied.


In [8]:
pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
probe = "Привет, мир! Hello, world! 🙂\n  spaces"

decoder = decoders.ByteLevel()
pieces = pre_tokenizer.pre_tokenize_str(probe)

[(decoder.decode([piece]), offsets) for piece, offsets in pieces]

[('Привет', (0, 6)),
 (',', (6, 7)),
 (' мир', (7, 11)),
 ('!', (11, 12)),
 (' Hello', (12, 18)),
 (',', (18, 19)),
 (' world', (19, 25)),
 ('!', (25, 26)),
 (' 🙂', (26, 28)),
 ('\n ', (28, 30)),
 (' spaces', (30, 37))]

## Train BPE

The complete 256-byte alphabet allows unseen characters to be represented. The EOS token is reserved but not automatically appended to documents. On a tiny corpus, the vocabulary may be smaller than the target. Streaming input does not eliminate the memory needed for BPE's corpus statistics.


In [11]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizer
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=2,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    special_tokens=["<|endoftext|>"],
)
tokenizer.train_from_iterator(texts, trainer=trainer, length=len(texts))
print("Learned vocabulary size:", tokenizer.get_vocab_size())





Learned vocabulary size: 8000


## Encode and decode

Inspect different scripts, emoji, and whitespace. The round-trip check uses ordinary text; decoding normally skips reserved special tokens.


In [13]:
examples = ["Привет, мир!", "Hello, world!", "🙂 café 中文", "  spaces\n\ttabs"]
for text in examples:
    encoding = tokenizer.encode(text)
    print(repr(text))
    print("Tokens:", [decoder.decode([token]) for token in encoding.tokens])
    print("IDs:", encoding.ids)
    print("Decoded:", repr(tokenizer.decode(encoding.ids)))
    assert tokenizer.decode(encoding.ids) == text
    print()


'Привет, мир!'
Tokens: ['Привет', ',', ' мир', '!']
IDs: [7626, 12, 2865, 1]
Decoded: 'Привет, мир!'

'Hello, world!'
Tokens: ['H', 'el', 'l', 'o', ',', ' w', 'orld', '!']
IDs: [40, 1476, 76, 79, 12, 6405, 4207, 1]
Decoded: 'Hello, world!'

'🙂 café 中文'
Tokens: ['�', '�', '�', ' c', 'a', 'f', '�', '�', ' ', '�', '�', '�', '�', '�', '�']
IDs: [6401, 248, 225, 2687, 65, 70, 128, 103, 221, 161, 117, 256, 163, 245, 230]
Decoded: '🙂 café 中文'

'  spaces\n\ttabs'
Tokens: [' ', ' s', 'p', 'ac', 'es', '\n', '\t', 't', 'ab', 's']
IDs: [221, 2688, 80, 2089, 2016, 199, 198, 84, 3966, 83]
Decoded: '  spaces\n\ttabs'



## Inspect vocabulary and merges

Compare early and late merge rules with the tokens above.


In [14]:
vocabulary = tokenizer.get_vocab()
print("First 20 tokens by ID:")
print(sorted(vocabulary.items(), key=lambda item: item[1])[:20])

merges = json.loads(tokenizer.to_str())["model"]["merges"]
print("Number of merges:", len(merges))
print("First 20 merges:", merges[:20])
print("Last 20 merges:", merges[-20:])


First 20 tokens by ID:
[('<|endoftext|>', 0), ('!', 1), ('"', 2), ('#', 3), ('$', 4), ('%', 5), ('&', 6), ("'", 7), ('(', 8), (')', 9), ('*', 10), ('+', 11), (',', 12), ('-', 13), ('.', 14), ('/', 15), ('0', 16), ('1', 17), ('2', 18), ('3', 19)]
Number of merges: 7743
First 20 merges: [['Ġ', 'Ð'], ['Ð', '¾'], ['Ð', 'µ'], ['Ð', '°'], ['Ñ', 'Ĥ'], ['Ð', '¸'], ['Ð', '½'], ['Ñ', 'ģ'], ['Ð¾', 'Ð'], ['Ñ', 'Ģ'], ['Ð', '»'], ['Ð', 'º'], ['Ñ', 'ĥ'], ['Ñ', 'ı'], ['Ðµ', 'Ð'], ['Ñ', 'ĭ'], ['Ð°', 'Ð'], ['ĠÐ', '¿'], ['Ñ', 'Į'], ['Ñģ', 'ÑĤ']]
Last 20 merges: [['Ñİ', 'Ñĩ'], ['Ñİ', 'Ð·'], ['Ñİ', 'Ð´Ð¸'], ['Ð¸ÑĤ', 'ÐµÑĢ'], ['Ð¸ÑĤ', 'Ð¾Ð½'], ['Ð¸ÑĤ', 'Ð½Ð¾Ð¹'], ['Ð¸ÑĤ', 'Ð°Ð»Ð°'], ['Ð¸ÑĤ', 'Ð°Ð¹ÑĤÐµ'], ['Ð¸ÑĤ', 'ÐµÐ»Ñİ'], ['ÑĪ', 'ÑĮ'], ['ÑĪ', 'Ð»Ð¸'], ['Ð¾Ð´', 'ÑĢ'], ['Ð¾Ð´', 'Ð¸Ð½'], ['Ð¾Ð´', 'Ð°ÑĤÐµÐ»Ñı'], ['Ð¾Ð´', 'Ð³Ð¾ÑĤÐ¾Ð²'], ['ÑĨ', 'Ð¸Ðº'], ['ÑĨ', 'Ð°Ñħ'], ['Ð·', 'ÐµÐ¼'], ['Ð·', 'Ð°Ð²'], ['Ð·', 'ÐµÐ½Ð¸ÑĤ']]


## Inspect held-out text

Stream ten documents from the separate test split, without training on them. Tokens per UTF-8 byte is a simple compression comparison when changing vocabulary size, not a model-quality score.


In [15]:
held_out = load_dataset(dataset_id, name=language, split="test", streaming=True)
evaluation_texts = [document["text"] for document in held_out.take(10)]
byte_count = sum(len(text.encode("utf-8")) for text in evaluation_texts)
token_count = sum(len(tokenizer.encode(text).ids) for text in evaluation_texts)

print("Held-out documents:", len(evaluation_texts))
print("Tokens:", token_count)
print("UTF-8 bytes:", byte_count)
if byte_count:
    print("Tokens per UTF-8 byte:", token_count / byte_count)


Resolving data files:   0%|          | 0/440 [00:00<?, ?it/s]

Held-out documents: 10
Tokens: 11740
UTF-8 bytes: 60766
Tokens per UTF-8 byte: 0.19320014481782577
